# MM-IMDb Neural Training Pipeline

Use this notebook to drive the neural training pipeline from one place: dataset inspection, split creation, model-selection training, optional XAI, and the local XAI dashboard.

For Google Colab, run the setup cell first. It can mount Google Drive, use a repository folder from Drive or clone the GitHub repository, install the notebook dependencies, and then let the rest of the pipeline run normally. Update the Colab path constants in the setup cell if your Drive folder or dataset location differs.

For a first pass, keep `RUN_MODE = "smoke"`. For a full thesis run, switch it to `"full"`, preferably on a GPU runtime, and review the controls before running the training cell.

## 1. Setup

In [ ]:
from __future__ import annotations

import copy
import json
import os
import subprocess
import sys
import time
from pathlib import Path
from pprint import pprint

IS_COLAB = "google.colab" in sys.modules

# Google Colab setup. These values are ignored when the notebook runs locally.
COLAB_MOUNT_GOOGLE_DRIVE = True
COLAB_PROJECT_ROOT = "/content/drive/MyDrive/Explaining-Predictions-of-Machine-Learning-Models"
COLAB_REPO_URL = "https://github.com/SamuelSulan/Explaining-Predictions-of-Machine-Learning-Models.git"
COLAB_CLONE_REPO_IF_MISSING = True
COLAB_INSTALL_DEPENDENCIES = True

# Optional Colab path overrides. Leave blank to use configs/default.yaml inside the repo.
# COLAB_DATA_ROOT may point either to a folder containing data/ or directly to the two dataset files.
COLAB_DATA_ROOT = "/content/drive/MyDrive/data"
COLAB_HDF5_PATH = ""
COLAB_METADATA_PATH = ""
COLAB_ARTICLE_PDF_PATH = ""

# Optional persistent output folder, for example /content/drive/MyDrive/mmimdb_outputs.
# Leave blank to write outputs/ inside the project folder.
COLAB_OUTPUT_DIR = ""

COLAB_PIP_PACKAGES = [
    "captum",
    "h5py",
    "iterative-stratification",
    "joblib",
    "matplotlib",
    "psutil",
    "pypdf",
    "pyyaml",
    "scikit-learn",
    "seaborn",
    "tqdm",
]


def run_checked(cmd: list[str], cwd: str | Path | None = None) -> None:
    print("Running:", " ".join(str(part) for part in cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


def find_local_project_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return current


def prepare_colab_project_root() -> Path:
    if COLAB_MOUNT_GOOGLE_DRIVE:
        from google.colab import drive

        drive.mount("/content/drive")

    project_root = Path(COLAB_PROJECT_ROOT).expanduser()
    if (project_root / "pyproject.toml").exists():
        return project_root.resolve()

    if COLAB_CLONE_REPO_IF_MISSING:
        if project_root.exists() and any(project_root.iterdir()):
            raise FileNotFoundError(
                f"COLAB_PROJECT_ROOT exists but does not look like this repo: {project_root}. "
                "Update COLAB_PROJECT_ROOT or set COLAB_CLONE_REPO_IF_MISSING = False."
            )
        project_root.parent.mkdir(parents=True, exist_ok=True)
        run_checked(["git", "clone", COLAB_REPO_URL, str(project_root)])
        return project_root.resolve()

    raise FileNotFoundError(
        f"Could not find pyproject.toml at COLAB_PROJECT_ROOT={project_root}. "
        "Upload/clone the repository there, or update COLAB_PROJECT_ROOT."
    )


PROJECT_ROOT = prepare_colab_project_root() if IS_COLAB else find_local_project_root()
os.chdir(PROJECT_ROOT)

if IS_COLAB and COLAB_INSTALL_DEPENDENCIES:
    # Colab already provides CUDA-enabled torch/torchvision on GPU runtimes, so this intentionally does not pin them.
    run_checked([sys.executable, "-m", "pip", "install", "-q", "-e", ".", *COLAB_PIP_PACKAGES], cwd=PROJECT_ROOT)

import numpy as np
import pandas as pd

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from mmimdb.data import DatasetPaths, dataset_label_stats, h5_summary, load_labels, load_metadata
from mmimdb.model_selection import run_training_process
from mmimdb.reporting import build_dataset_report
from mmimdb.splits import load_split_indices, save_splits
from mmimdb.utils import load_config, load_json, resolve_path

print(f"Runtime: {'Google Colab' if IS_COLAB else 'local'}")
print(f"Project root: {PROJECT_ROOT}")
print(f"Python: {sys.executable}")

## 2. Control Panel

In [ ]:
CONFIG_PATH = "configs/default.yaml"

# Main switch. Use "smoke" for a quick end-to-end check, "full" for the real run.
RUN_MODE = "smoke"  # "smoke" or "full"

# Pipeline steps.
RUN_INSPECTION = True
CREATE_OR_REFRESH_SPLITS = True
BUILD_DATASET_REPORT = False if IS_COLAB else True
TRAIN_MODELS = True
RUN_XAI_AFTER_TRAINING = False
START_XAI_DASHBOARD = False

# Training controls.
MODEL_TYPE = "neural"  # fixed for this notebook
FULL_FOLDS = None  # None uses configs/default.yaml

# Smoke-test controls. These keep neural training lightweight on CPU.
SMOKE_LIMIT = 12
SMOKE_FOLDS = 2
SMOKE_EPOCHS = 1
SMOKE_BATCH_SIZE = 2
SMOKE_NO_PRETRAINED_IMAGE = True

# XAI controls.
XAI_MODEL_TYPE = "neural"  # fixed for this notebook
XAI_SPLIT = "test"
XAI_LIMIT = 5 if RUN_MODE == "smoke" else 10
XAI_TARGET_POLICY = "predicted"  # "top", "top_k", "predicted", or "all"
XAI_TARGET_GENRES = ""  # comma-separated explicit genres, e.g. "Crime,Drama"
XAI_N_STEPS = 2 if RUN_MODE == "smoke" else None
XAI_TOP_K = 8
XAI_IMAGE_OCCLUSION_GRID = 2 if RUN_MODE == "smoke" else None
XAI_NO_OCCLUSION = RUN_MODE == "smoke"

# Dashboard controls. The dashboard is meant for local runs; Colab cannot expose it directly without a tunnel.
DASHBOARD_HOST = "127.0.0.1"
DASHBOARD_PORT = 8050

assert RUN_MODE in {"smoke", "full"}
assert MODEL_TYPE in {"classic", "neural", "both"}
assert XAI_MODEL_TYPE in {"classic", "neural", "both"}
print(f"Neural notebook configured for {RUN_MODE!r} mode.")

## 3. Load Config and Apply Notebook Overrides

In [ ]:
base_config = load_config(CONFIG_PATH)
config = copy.deepcopy(base_config)


def nonempty_path(value: str) -> str | None:
    value = str(value).strip()
    return value or None


def first_existing_or_default(*candidates: Path) -> Path:
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return candidates[0]


def apply_colab_overrides(config: dict) -> None:
    if not IS_COLAB:
        return

    data_root_value = nonempty_path(COLAB_DATA_ROOT)
    if data_root_value:
        data_root = Path(data_root_value).expanduser()
        config["paths"]["hdf5"] = str(
            first_existing_or_default(
                data_root / "data" / "multimodal_imdb.hdf5",
                data_root / "multimodal_imdb.hdf5",
            )
        )
        config["paths"]["metadata"] = str(
            first_existing_or_default(
                data_root / "data" / "metadata.npy",
                data_root / "metadata.npy",
            )
        )
        article_pdf = first_existing_or_default(
            data_root / "Article about dataset and how was it used.pdf",
            data_root / "data" / "Article about dataset and how was it used.pdf",
        )
        config["paths"]["article_pdf"] = str(article_pdf)

    if nonempty_path(COLAB_HDF5_PATH):
        config["paths"]["hdf5"] = nonempty_path(COLAB_HDF5_PATH)
    if nonempty_path(COLAB_METADATA_PATH):
        config["paths"]["metadata"] = nonempty_path(COLAB_METADATA_PATH)
    if nonempty_path(COLAB_ARTICLE_PDF_PATH):
        config["paths"]["article_pdf"] = nonempty_path(COLAB_ARTICLE_PDF_PATH)

    output_dir_value = nonempty_path(COLAB_OUTPUT_DIR)
    if output_dir_value:
        output_root = Path(output_dir_value).expanduser()
        config.setdefault("project", {})["output_dir"] = str(output_root)
        config.setdefault("splits", {})["output_dir"] = str(output_root / "splits")
        config.setdefault("training", {})["output_dir"] = str(output_root / "model_selection")
        config.setdefault("model_registry", {})["best_dir"] = str(output_root / "models" / "best")


apply_colab_overrides(config)

training_limit = None
folds_override = FULL_FOLDS

if RUN_MODE == "smoke":
    training_limit = SMOKE_LIMIT
    folds_override = SMOKE_FOLDS
    config.setdefault("neural", {})["epochs"] = SMOKE_EPOCHS
    config.setdefault("neural", {})["batch_size"] = SMOKE_BATCH_SIZE
    if SMOKE_NO_PRETRAINED_IMAGE:
        config.setdefault("neural", {})["pretrained_image"] = False

paths = DatasetPaths.from_config(config)
split_dir = resolve_path(config["splits"]["output_dir"])
training_output_dir = resolve_path(config.get("training", {}).get("output_dir", "outputs/model_selection"))

print("Dataset paths:")
pprint(paths)
print(f"Split dir: {split_dir}")
print(f"Training output dir: {training_output_dir}")

## 4. Preflight and Dataset Inspection

### Optional Kaggle Dataset Download

This cell checks the configured dataset paths. If the HDF5 or metadata file is missing, it downloads `johnarevalo/mmimdb` from Kaggle and places the required files beside the configured target paths. Kaggle API credentials are required through `~/.kaggle/kaggle.json` or `KAGGLE_USERNAME` / `KAGGLE_KEY`.

In Colab, you can either set `COLAB_DATA_ROOT` / `COLAB_HDF5_PATH` / `COLAB_METADATA_PATH` in the setup cell to point at files on mounted Drive, or let this cell download the dataset after you configure Kaggle credentials.

In [ ]:
from importlib.util import find_spec
import shutil

DATASET_SLUG = "johnarevalo/mmimdb"
DATASET_PAGE = "https://www.kaggle.com/datasets/johnarevalo/mmimdb/data"
EXPECTED_DATASET_FILES = {
    "multimodal_imdb.hdf5": Path(paths.hdf5),
    "metadata.npy": Path(paths.metadata),
}
DATASET_COMMON_DIR = Path(os.path.commonpath([str(path.parent) for path in EXPECTED_DATASET_FILES.values()]))
DATASET_DIR = DATASET_COMMON_DIR.parent if DATASET_COMMON_DIR.name == "data" else DATASET_COMMON_DIR
KAGGLE_DOWNLOAD_DIR = DATASET_DIR / "_kaggle_download"


def dataset_files_ready() -> bool:
    return all(path.exists() for path in EXPECTED_DATASET_FILES.values())


def find_downloaded_dataset_file(filename: str) -> Path | None:
    target = EXPECTED_DATASET_FILES[filename].resolve()
    candidates = [path for path in DATASET_DIR.rglob(filename) if path.resolve() != target]
    return sorted(candidates, key=lambda path: len(path.parts))[0] if candidates else None


def ensure_kaggle_package() -> None:
    if find_spec("kaggle") is None:
        print("Installing Kaggle API package...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kaggle"], check=True)


def download_mmimdb_from_kaggle() -> None:
    ensure_kaggle_package()
    KAGGLE_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi

        api = KaggleApi()
        api.authenticate()
        api.dataset_download_files(DATASET_SLUG, path=str(KAGGLE_DOWNLOAD_DIR), unzip=True, quiet=False)
    except Exception as exc:
        raise RuntimeError(
            "Could not download the MM-IMDb dataset from Kaggle. "
            "Open the dataset page and configure Kaggle API credentials first: "
            f"{DATASET_PAGE}"
        ) from exc


def place_dataset_files() -> None:
    for filename, target in EXPECTED_DATASET_FILES.items():
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            continue
        source = find_downloaded_dataset_file(filename)
        if source is None:
            continue
        print(f"Moving {source} -> {target}")
        shutil.move(str(source), str(target))


place_dataset_files()
if not dataset_files_ready():
    missing = [str(path) for path in EXPECTED_DATASET_FILES.values() if not path.exists()]
    print("Missing dataset files:", missing)
    print(f"Downloading {DATASET_SLUG} from Kaggle...")
    download_mmimdb_from_kaggle()
    place_dataset_files()

if not dataset_files_ready():
    missing = [str(path) for path in EXPECTED_DATASET_FILES.values() if not path.exists()]
    raise FileNotFoundError("Dataset download finished, but required files are still missing: " + ", ".join(missing))

print("Dataset files are ready:")
for path in EXPECTED_DATASET_FILES.values():
    print(f"- {path}")

In [ ]:
required_files = [paths.hdf5, paths.metadata]
missing = [path for path in required_files if not Path(path).exists()]
if missing:
    raise FileNotFoundError("Missing dataset files: " + ", ".join(str(path) for path in missing))

print("Required dataset files are present.")

if RUN_INSPECTION:
    print("\nHDF5 summary")
    display(pd.DataFrame(h5_summary(paths.hdf5)))

    metadata = load_metadata(paths.metadata)
    print("\nMetadata")
    print("keys:", sorted(metadata.keys()))
    print("vocab_size:", metadata.get("vocab_size"))
    if "lookup" in metadata:
        print("lookup shape:", metadata["lookup"].shape)

    y = load_labels(paths.hdf5)
    print("\nLabel stats")
    pprint(dataset_label_stats(y))
else:
    print("Inspection skipped.")

## 5. Create or Load Splits

In [ ]:
split_files = [split_dir / "train_indices.npy", split_dir / "val_indices.npy", split_dir / "test_indices.npy"]

if CREATE_OR_REFRESH_SPLITS or not all(path.exists() for path in split_files):
    split_metadata = save_splits(
        paths.hdf5,
        split_dir,
        train_size=float(config["splits"]["train_size"]),
        val_size=float(config["splits"]["val_size"]),
        test_size=float(config["splits"]["test_size"]),
        random_state=int(config["splits"]["random_state"]),
    )
    print("Saved fresh splits.")
    pprint(split_metadata)
else:
    print("Using existing splits.")

train_idx, val_idx, test_idx = load_split_indices(split_dir)
pd.DataFrame(
    [
        {"split": "train", "count": len(train_idx)},
        {"split": "val", "count": len(val_idx)},
        {"split": "test", "count": len(test_idx)},
    ]
)

## 6. Build Dataset Report

In [ ]:
if BUILD_DATASET_REPORT:
    report_path = build_dataset_report(
        paths.hdf5,
        paths.metadata,
        paths.article_pdf,
        output_path="docs/dataset_preprocessing_report.md",
        figures_dir=resolve_path(config["project"]["output_dir"]) / "figures",
        split_metadata_path=split_dir / "split_metadata.json",
    )
    print(f"Report written to: {report_path}")
else:
    print("Dataset report skipped.")

## 7. Train and Select Models

In [ ]:
training_summary = None

if TRAIN_MODELS:
    started = time.perf_counter()
    training_summary = run_training_process(
        config,
        model_type=MODEL_TYPE,
        limit=training_limit,
        folds_override=folds_override,
    )
    elapsed_minutes = (time.perf_counter() - started) / 60
    print(f"Training finished in {elapsed_minutes:.2f} minutes.")
    print("Overall best:")
    pprint(training_summary.get("overall_best"))
else:
    summary_path = training_output_dir / "training_summary.json"
    if summary_path.exists():
        training_summary = load_json(summary_path)
        print(f"Loaded existing summary: {summary_path}")
    else:
        print("Training skipped and no existing training summary was found.")

## 8. Review Training Results

In [ ]:
summary_path = training_output_dir / "training_summary.json"
if training_summary is None and summary_path.exists():
    training_summary = load_json(summary_path)

def candidate_table(summary: dict, kind: str) -> pd.DataFrame:
    kind_summary = summary.get("kinds", {}).get(kind, {})
    rows = []
    for candidate in kind_summary.get("candidates", []):
        rows.append(
            {
                "kind": kind,
                "candidate": candidate.get("name"),
                "status": candidate.get("status"),
                "mean_score": candidate.get("mean_score"),
                "std_score": candidate.get("std_score"),
                "folds_completed": candidate.get("folds_completed"),
            }
        )
    return pd.DataFrame(rows)

if training_summary:
    print(f"Summary path: {summary_path}")
    print("Overall best:")
    pprint(training_summary.get("overall_best"))

    tables = [candidate_table(training_summary, kind) for kind in ["classic", "neural"]]
    tables = [table for table in tables if not table.empty]
    if tables:
        display(pd.concat(tables, ignore_index=True).sort_values(["kind", "mean_score"], ascending=[True, False]))
else:
    print("No training summary available yet.")

## 9. Optional XAI

In [ ]:
def run_project_script(script: str, *args: object) -> subprocess.CompletedProcess:
    cmd = [sys.executable, str(PROJECT_ROOT / script), *(str(arg) for arg in args)]
    print("Running:", " ".join(cmd))
    return subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

def final_model_path(summary: dict | None, kind: str) -> str | None:
    if not summary:
        return None
    return summary.get("kinds", {}).get(kind, {}).get("final", {}).get("model_path")

if RUN_XAI_AFTER_TRAINING:
    xai_args: list[object] = [
        "--config", CONFIG_PATH,
        "--model-type", XAI_MODEL_TYPE,
        "--split", XAI_SPLIT,
        "--limit", XAI_LIMIT,
        "--target-policy", XAI_TARGET_POLICY,
        "--top-k", XAI_TOP_K,
    ]
    if XAI_TARGET_GENRES:
        xai_args.extend(["--target-genres", XAI_TARGET_GENRES])
    if XAI_N_STEPS is not None:
        xai_args.extend(["--n-steps", XAI_N_STEPS])
    if XAI_IMAGE_OCCLUSION_GRID is not None:
        xai_args.extend(["--image-occlusion-grid", XAI_IMAGE_OCCLUSION_GRID])
    if XAI_NO_OCCLUSION:
        xai_args.append("--no-occlusion")

    classic_model_path = final_model_path(training_summary, "classic")
    neural_model_path = final_model_path(training_summary, "neural")
    if XAI_MODEL_TYPE in {"classic", "both"} and classic_model_path:
        xai_args.extend(["--classic-model", classic_model_path])
    if XAI_MODEL_TYPE in {"neural", "both"} and neural_model_path:
        xai_args.extend(["--neural-checkpoint", neural_model_path])

    run_project_script("scripts/run_xai.py", *xai_args)
else:
    print("XAI skipped.")

## 10. Optional XAI Dashboard

In [ ]:
dashboard_process = None

if START_XAI_DASHBOARD:
    dashboard_process = subprocess.Popen(
        [
            sys.executable,
            str(PROJECT_ROOT / "scripts/xai_dashboard.py"),
            "--host", DASHBOARD_HOST,
            "--port", str(DASHBOARD_PORT),
        ],
        cwd=PROJECT_ROOT,
    )
    print(f"Dashboard started at http://{DASHBOARD_HOST}:{DASHBOARD_PORT}")
    print("Stop it later with: dashboard_process.terminate()")
else:
    print("Dashboard not started.")

## 11. Useful One-Off Commands

In [ ]:
# Uncomment a command when you want to run a specific script manually.

# run_project_script("scripts/inspect_dataset.py", "--config", CONFIG_PATH)
# run_project_script("scripts/create_splits.py", "--config", CONFIG_PATH)
# run_project_script("scripts/train_models.py", "--config", CONFIG_PATH, "--model-type", "classic", "--limit", 100, "--folds", 2)
# run_project_script("scripts/train_models.py", "--config", CONFIG_PATH, "--model-type", "both")
# run_project_script("scripts/run_xai.py", "--config", CONFIG_PATH, "--model-type", "both", "--limit", 10, "--no-occlusion")